In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!pip install xgboost lightgbm "mlflow<3"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.1/40.1 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 29.0/29.0 MB 68.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.3/6.3 MB 105.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 147.8/147.8 kB 11.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 8.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.0/85.0 kB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 774.7/774.7 kB 47.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.3/207.3 kB 17.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.5/65.5 kB 5.3 MB/s eta 0:00:00
  Attempting uninstall: packaging
    Found existing installation: packaging 25.0
    Uninstalling packaging-25.0:
      Successfully uninstalled packaging-25.0
  Attempting uninstall: cachetools
    Found existing installation: cachetools 6.2.2
    Uninstalling cachetools-6.2.2:
   

In [1]:
base_folder = "/content/drive/MyDrive/Colab Notebooks/airline_passenger_satisfaction"
%cd "{base_folder}"

/content/drive/MyDrive/Colab Notebooks/airline_passenger_satisfaction


In [2]:
import sqlite3
import pandas as pd

conn = sqlite3.connect(f"{base_folder}/data/airline.db")

airline = pd.read_sql_query(
    """
    SELECT
        p.passenger_id,
        p.gender,
        p.age,
        p.customer_type,
        p.travel_type,
        p.class,
        f.flight_distance,
        f.departure_delay,
        f.arrival_delay,
        sr.inflight_wifi,
        sr.time_convenient,
        sr.online_booking,
        sr.gate_location,
        sr.food_drink,
        sr.online_boarding,
        sr.seat_comfort,
        sr.inflight_entertainment,
        sr.onboard_service,
        sr.legroom,
        sr.baggage,
        sr.checkin,
        sr.inflight_service,
        sr.cleanliness,
        sr.satisfaction
    FROM passenger AS p
    JOIN service_ratings AS sr
        ON sr.passenger_id = p.passenger_id
    JOIN flight AS f
        ON f.flight_id = sr.flight_id
    ORDER BY p.passenger_id
    """,
    conn,
)

conn.close()

airline.head()

,passenger_id,gender,age,customer_type,travel_type,class,flight_distance,departure_delay,arrival_delay,inflight_wifi,...,online_boarding,seat_comfort,inflight_entertainment,onboard_service,legroom,baggage,checkin,inflight_service,cleanliness,satisfaction
0,1,Male,13,Loyal Customer,Personal Travel,Eco Plus,460,25,18.0,3,...,3,5,5,4,3,4,4,5,5,neutral or dissatisfied
1,2,Male,25,disloyal Customer,Business travel,Business,235,1,6.0,3,...,3,1,1,1,5,3,1,4,1,neutral or dissatisfied
2,3,Female,26,Loyal Customer,Business travel,Business,1142,0,0.0,2,...,5,5,5,4,3,4,4,4,5,satisfied
3,4,Female,25,Loyal Customer,Business travel,Business,562,11,9.0,2,...,2,2,2,2,5,3,1,4,2,neutral or dissatisfied
4,5,Male,61,Loyal Customer,Business travel,Business,214,0,0.0,3,...,5,5,3,3,4,4,3,3,3,satisfied


In [3]:
# =============================================================================
# FULL PIPELINE (AIRLINE CLASSIFICATION):
# - Encode string labels
# - Build preprocessing
# - Stratified train/test split
# - Train & log 4 models WITHOUT PCA
# - Train & log 4 models WITH PCA
# - Pick GLOBAL best among 8 models by Test F1
# - Save the global best model
# =============================================================================

import os
import time
import numpy as np
import pandas as pd

from dotenv import load_dotenv

from sklearn.decomposition import PCA
from sklearn.metrics import f1_score, make_scorer
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import LabelEncoder

import mlflow
from mlflow.models import infer_signature
import joblib

# Import shared airline pipeline components
from airline_pipeline import (
    build_preprocessing,
    make_estimator_for_name,
)

start_time = time.monotonic()

# =============================================================================
# STEP 1: Build Full ML Preprocessing Pipeline
# =============================================================================
preprocessing = build_preprocessing()
print("✓ STEP 1: Preprocessing pipeline created.")

# =============================================================================
# STEP 2: Encode string labels & Stratified Train/Test Split
# =============================================================================
airline = airline.dropna()
X = airline.drop(["satisfaction", "passenger_id"], axis=1).copy()
y = airline["satisfaction"].copy()

# Encode target labels: 0 = 'neutral or dissatisfied', 1 = 'satisfied'
le = LabelEncoder()
y_enc = le.fit_transform(y)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y_enc,
    test_size=0.20,
    stratify=y_enc,
    random_state=42,
)

print(f"✓ STEP 2: Stratified split done. Train size: {len(X_train)}, Test size: {len(X_test)}")

# Define F1 scorer
f1_scorer = make_scorer(f1_score, average='binary', pos_label=1)

# =============================================================================
# STEP 3: Define 4 Model Pipelines (WITHOUT PCA)
# =============================================================================
models = {}
for name in ["ridge", "histgradientboosting", "xgboost", "lightgbm"]:
    est = make_estimator_for_name(name)
    models[name] = make_pipeline(preprocessing, est)

print("✓ STEP 3: 4 baseline airline classifier pipelines defined.")

# =============================================================================
# STEP 4: Configure MLflow
# =============================================================================
load_dotenv(
    dotenv_path="/content/drive/MyDrive/Colab Notebooks/airline_passenger_satisfaction/notebooks/.env",
    override=True
)
MLFLOW_TRACKING_URI = os.getenv("MLFLOW_TRACKING_URI")
MLFLOW_TRACKING_USERNAME = os.getenv("MLFLOW_TRACKING_USERNAME")
MLFLOW_TRACKING_PASSWORD = os.getenv("MLFLOW_TRACKING_PASSWORD")

if MLFLOW_TRACKING_USERNAME:
    os.environ["MLFLOW_TRACKING_USERNAME"] = MLFLOW_TRACKING_USERNAME
if MLFLOW_TRACKING_PASSWORD:
    os.environ["MLFLOW_TRACKING_PASSWORD"] = MLFLOW_TRACKING_PASSWORD

mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
mlflow.set_experiment("airline_satisfaction_classification")

print("✓ STEP 4: MLflow configured.")

# =============================================================================
# STEP 5: Train, Evaluate, and Log 4 Baseline Models (NO PCA)
# =============================================================================
results = {}

for name, pipeline in models.items():
    print("\n" + "=" * 80)
    print(f"Training baseline classifier: {name}")
    print("=" * 80)

    # CV F1
    cv_scores = cross_val_score(
        pipeline,
        X_train,
        y_train,
        cv=3,
        scoring=f1_scorer,
        n_jobs=-1,
    )
    cv_f1 = cv_scores.mean() * 100  # convert to 0-100
    print(f"{name} (no PCA) CV F1: {cv_f1:.2f}")

    # Fit on full training set
    pipeline.fit(X_train, y_train)

    # Test F1
    y_pred = pipeline.predict(X_test)
    test_f1 = f1_score(y_test, y_pred, average='binary', pos_label=1) * 100  # convert to 0-100
    print(f"{name} (no PCA) Test F1: {test_f1:.2f}")

    results[name] = {
        "pipeline": pipeline,
        "cv_f1": cv_f1,
        "test_f1": test_f1,
    }

    # Log to MLflow
    with mlflow.start_run(run_name=f"{name}_baseline"):
        mlflow.log_param("model_family", name)
        mlflow.log_param("uses_pca", False)

        est_step = list(pipeline.named_steps.keys())[-1]
        mlflow.log_params({
            f"{est_step}__{k}": v
            for k, v in pipeline.named_steps[est_step].get_params().items()
        })

        mlflow.log_metric("cv_F1", cv_f1)
        mlflow.log_metric("test_F1", test_f1)

        signature = infer_signature(X_train, pipeline.predict(X_train))
        mlflow.sklearn.log_model(
            pipeline,
            artifact_path="airline_classifier",
            signature=signature,
            input_example=X_train,
        )

print("\n✓ STEP 5: All baseline classifiers trained and logged.")

# =============================================================================
# STEP 6: Train, Evaluate, and Log PCA Versions of ALL 4 Models
# =============================================================================
pca_results = {}

for name in models.keys():
    print("\n" + "=" * 80)
    print(f"Training PCA classifier: {name}")
    print("=" * 80)

    est = make_estimator_for_name(name)
    pca_pipeline = make_pipeline(
        preprocessing,
        PCA(n_components=0.95),
        est,
    )

    cv_scores = cross_val_score(
        pca_pipeline,
        X_train,
        y_train,
        cv=3,
        scoring=f1_scorer,
        n_jobs=-1,
    )
    cv_f1 = cv_scores.mean() * 100  # convert to 0-100
    print(f"{name}_with_pca CV F1: {cv_f1:.2f}")

    pca_pipeline.fit(X_train, y_train)

    y_pred = pca_pipeline.predict(X_test)
    test_f1 = f1_score(y_test, y_pred, average='binary', pos_label=1) * 100  # convert to 0-100

    model_key = f"{name}_with_pca"
    pca_results[model_key] = {
        "pipeline": pca_pipeline,
        "cv_f1": cv_f1,
        "test_f1": test_f1,
    }

    print(f"{model_key} Test F1: {test_f1:.2f}")

    with mlflow.start_run(run_name=model_key):
        mlflow.log_param("model_family", name)
        mlflow.log_param("uses_pca", True)
        mlflow.log_param("pca__n_components", 0.95)
        mlflow.log_metric("cv_F1", cv_f1)
        mlflow.log_metric("test_F1", test_f1)

print("\n✓ STEP 6: All PCA classifiers trained and logged.")

# =============================================================================
# STEP 7: Choose GLOBAL Best Model (F1)
# =============================================================================
all_results = {**results, **pca_results}

# Use max() for F1
global_best_name = max(all_results, key=lambda k: all_results[k]["test_f1"])
global_best_f1 = all_results[global_best_name]["test_f1"]
global_best_cv_f1 = all_results[global_best_name]["cv_f1"]
global_best_pipeline = all_results[global_best_name]["pipeline"]
uses_pca = "with_pca" in global_best_name

print("\n" + "=" * 80)
print("GLOBAL BEST AIRLINE CLASSIFIER")
print("=" * 80)
print(f"Global best model key: {global_best_name}")
print(f"Global best CV F1:    {global_best_cv_f1:.2f}")
print(f"Global best Test F1:  {global_best_f1:.2f}")
print(f"Uses PCA:             {uses_pca}")

# =============================================================================
# SAVE ALL MODELS
# =============================================================================
for model_name, info in all_results.items():
    model_path = f"{base_folder}/models/all_models/without_optuna/{model_name}.pkl"
    joblib.dump(info["pipeline"], model_path)

print("✓ All trained models saved.")

# =============================================================================
# SAVE ALL METRICS
# =============================================================================
metrics_records = []
for model_name, info in all_results.items():
    metrics_records.append({
        "model_name": model_name,
        "cv_f1": info["cv_f1"],
        "test_f1": info["test_f1"],
        "uses_pca": "with_pca" in model_name,
        "tuned": "optuna" in model_name
    })
metrics_df = pd.DataFrame(metrics_records)
metrics_path = f"{base_folder}/metrics/without_optuna/all_f1_scores.csv"
metrics_df.to_csv(metrics_path, index=False)

print(f"✓ All metrics saved to {metrics_path}")

# =============================================================================
# STEP 8: Save GLOBAL Best Model
# =============================================================================
def save_model(model, filename="global_best_airline_classifier.pkl"):
    joblib.dump(model, filename)
    print(f"✓ Model saved to {filename}")

print("\n" + "-" * 80)
print("Saving GLOBAL best model...")
print("-" * 80)

save_model(global_best_pipeline, filename=f"{base_folder}/models/global_best_airline_classifier.pkl")
print("✓ Global best airline classifier saved.")

end_time = time.monotonic()
elapsed_time = end_time - start_time
minutes = int(elapsed_time // 60)
seconds = elapsed_time % 60
print(f"Elapsed time: {minutes} minutes and {seconds:.2f} seconds")


✓ STEP 1: Preprocessing pipeline created.
✓ STEP 2: Stratified split done. Train size: 103589, Test size: 25898
✓ STEP 3: 4 baseline airline classifier pipelines defined.


2025/12/18 00:09:23 INFO mlflow.tracking.fluent: Experiment with name 'airline_satisfaction_classification' does not exist. Creating a new experiment.


✓ STEP 4: MLflow configured.

Training baseline classifier: ridge
ridge (no PCA) CV F1: 84.88
ridge (no PCA) Test F1: 85.07


/usr/local/lib/python3.12/dist-packages/mlflow/types/utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(


🏃 View run ridge_baseline at: https://dagshub.com/dharshinivenkata/airline_passenger_satisfaction.mlflow/#/experiments/0/runs/2ee7b176173a45aca0134b380ec260fa
🧪 View experiment at: https://dagshub.com/dharshinivenkata/airline_passenger_satisfaction.mlflow/#/experiments/0

Training baseline classifier: histgradientboosting
histgradientboosting (no PCA) CV F1: 95.40
histgradientboosting (no PCA) Test F1: 95.43


...

lightgbm (no PCA) Test F1: 95.84


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/mlflow/types/utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/vali

🏃 View run lightgbm_baseline at: https://dagshub.com/dharshinivenkata/airline_passenger_satisfaction.mlflow/#/experiments/0/runs/b9d411adbfb241febab5d757ce6f277f
🧪 View experiment at: https://dagshub.com/dharshinivenkata/airline_passenger_satisfaction.mlflow/#/experiments/0

✓ STEP 5: All baseline classifiers trained and logged.

Training PCA classifier: ridge
ridge_with_pca CV F1: 81.57
ridge_with_pca Test F1: 82.08
🏃 View run ridge_with_pca at: https://dagshub.com/dharshinivenkata/airline_passenger_satisfaction.mlflow/#/experiments/0/runs/100830f819794b6a93895a7be36218b4
🧪 View experiment at: https://dagshub.com/dharshinivenkata/airline_passenger_satisfaction.mlflow/#/experiments/0

Training PCA classifier: histgradientboosting
histgradientboosting_with_pca CV F1: 90.02
histgradientboosting_with_pca Test F1: 90.23
🏃 View run histgradientboosting_with_pca at: https://dagshub.com/dharshinivenkata/airline_passenger_satisfaction.mlflow/#/experiments/0/runs/7bb5ce03e345448a8fa454c62765dde

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


lightgbm_with_pca Test F1: 91.56
🏃 View run lightgbm_with_pca at: https://dagshub.com/dharshinivenkata/airline_passenger_satisfaction.mlflow/#/experiments/0/runs/fe1116f8520c4336885c8b70897c2d36
🧪 View experiment at: https://dagshub.com/dharshinivenkata/airline_passenger_satisfaction.mlflow/#/experiments/0

✓ STEP 6: All PCA classifiers trained and logged.

GLOBAL BEST AIRLINE CLASSIFIER
Global best model key: lightgbm
Global best CV F1:    95.83
Global best Test F1:  95.84
Uses PCA:             False
✓ All trained models saved.
✓ All metrics saved to /content/drive/MyDrive/Colab Notebooks/airline_passenger_satisfaction/metrics/without_optuna/all_f1_scores.csv

--------------------------------------------------------------------------------
Saving GLOBAL best model...
--------------------------------------------------------------------------------
✓ Model saved to /content/drive/MyDrive/Colab Notebooks/airline_passenger_satisfaction/models/global_best_airline_classifier.pkl
✓ Global be